# Data Visualization Project — Reproducible Notebook

Run this notebook from top to bottom using **Run All**.

Place these files in the same folder as this notebook:
- `crim_off_cat$defaultview_spreadsheet.xlsx`
- `demo_pjan$defaultview_spreadsheet.xlsx`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## File paths

In [ ]:
crime_path = Path("crim_off_cat$defaultview_spreadsheet.xlsx")
pop_path = Path("demo_pjan$defaultview_spreadsheet.xlsx")

print("Crime file exists:", crime_path.exists())
print("Population file exists:", pop_path.exists())

## Load raw Excel files

In [ ]:
crime_raw = pd.read_excel(crime_path, header=None)
pop_raw = pd.read_excel(pop_path, header=None)

crime_raw.head()

## Clean crime data

In [ ]:
crime_data = crime_raw.iloc[10:].copy()
crime_cols = [0] + list(range(1, crime_data.shape[1], 2))
crime_data = crime_data.iloc[:, crime_cols]

crime_years = crime_raw.iloc[8].dropna().tolist()
crime_years = [str(y).strip() for y in crime_years if str(y).strip() != "TIME"]
crime_years = crime_years[:crime_data.shape[1] - 1]

crime_data.columns = ["Country"] + crime_years
crime_data["Country"] = crime_data["Country"].astype(str).str.strip()

crime_data = crime_data[~crime_data["Country"].isin([
    "Special value", ":", "Dataset:", "Dataset: ", "Last updated:", "Time frequency",
    "International classification of crime for statistical purposes (ICCS)",
    "Unit of measure", "TIME", "GEO (Labels)"
])]

crime_data = crime_data.set_index("Country")
crime_data = crime_data.apply(pd.to_numeric, errors="coerce")
crime_data.head()

## Clean population data

In [ ]:
pop_data = pop_raw.iloc[11:].copy()
pop_data = pop_data[
    ~pop_data.iloc[:, 0].astype(str).str.contains("European|Euro area", na=False)
].copy()

pop_cols = [0] + list(range(1, pop_data.shape[1], 2))
pop_data = pop_data.iloc[:, pop_cols]

pop_years = pop_raw.iloc[9].dropna().tolist()
pop_years = [str(y).strip() for y in pop_years if str(y).strip() != "TIME"]
pop_years = pop_years[:pop_data.shape[1] - 1]

pop_data.columns = ["Country"] + pop_years
pop_data["Country"] = pop_data["Country"].astype(str).str.strip()

pop_data = pop_data.set_index("Country")
pop_data = pop_data.apply(pd.to_numeric, errors="coerce")
pop_data.head()

## Filter to EU-27 and align

In [ ]:
eu_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czechia",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia",
    "Slovenia", "Spain", "Sweden"
]

crime_eu = crime_data.loc[crime_data.index.isin(eu_countries)].copy()
pop_eu = pop_data.loc[pop_data.index.isin(eu_countries)].copy()

common_countries = crime_eu.index.intersection(pop_eu.index)

crime_eu = crime_eu.loc[common_countries].copy()
pop_eu = pop_eu.loc[common_countries].copy()

print("Common countries:", len(common_countries))
print(common_countries.tolist())

## Determine years

In [ ]:
crime_eu.columns = crime_eu.columns.astype(str)
pop_eu.columns = pop_eu.columns.astype(str)

year_cols = [c for c in crime_eu.columns if c.isdigit()]
start_year = min(year_cols)
latest_year = max(year_cols)

print("Start year:", start_year)
print("Latest year:", latest_year)

## Baseline ranking

In [ ]:
crime_latest = pd.to_numeric(crime_eu[latest_year], errors="coerce").groupby(level=0).sum()

ranking_table = crime_latest.sort_values(ascending=False).reset_index()
ranking_table.columns = ["Country", "Raw Crime Value"]
ranking_table["Raw Rank"] = range(1, len(ranking_table) + 1)

ranking_table

## Per-capita comparison

In [ ]:
population_latest = pd.to_numeric(pop_eu[latest_year], errors="coerce").groupby(level=0).first()

comparison_table = ranking_table.copy().set_index("Country")
comparison_table[f"Population_{latest_year}"] = population_latest

comparison_table["Crime_per_100k"] = (
    comparison_table["Raw Crime Value"] / comparison_table[f"Population_{latest_year}"]
) * 100000

comparison_table["PerCapita Rank"] = (
    comparison_table["Crime_per_100k"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

comparison_table = comparison_table.sort_values("PerCapita Rank")
comparison_table

## Trend table

In [ ]:
crime_start = pd.to_numeric(crime_eu[start_year], errors="coerce").groupby(level=0).sum()

trend_pct = ((crime_latest - crime_start) / crime_start.replace(0, np.nan)) * 100
trend_pct = trend_pct.replace([np.inf, -np.inf], np.nan).dropna()

trend_table = trend_pct.reset_index()
trend_table.columns = ["Country", "Trend_%"]
trend_table = trend_table.groupby("Country", as_index=False)["Trend_%"].mean()
trend_table = trend_table.sort_values("Trend_%", ascending=False).reset_index(drop=True)

trend_table

## Profile table

In [ ]:
per_capita_table = comparison_table.reset_index()[["Country", "Crime_per_100k", "PerCapita Rank"]].copy()

profile_table = pd.merge(
    ranking_table,
    per_capita_table,
    on="Country",
    how="inner"
)

profile_table = pd.merge(
    profile_table,
    trend_table,
    on="Country",
    how="inner"
)

profile_table = profile_table.sort_values("Crime_per_100k", ascending=False).reset_index(drop=True)
profile_table

## Export final tables

In [ ]:
ranking_table.to_csv("eu_crime_raw_ranking_full.csv", index=False)
comparison_table.reset_index().to_csv("eu_crime_per_capita_full.csv", index=False)
trend_table.to_csv("eu_crime_trend_full.csv", index=False)
profile_table.to_csv("eu_crime_profile_full.csv", index=False)

with pd.ExcelWriter("eu_crime_project_tables.xlsx", engine="openpyxl") as writer:
    ranking_table.to_excel(writer, sheet_name="Raw_Ranking", index=False)
    comparison_table.reset_index().to_excel(writer, sheet_name="Per_Capita", index=False)
    trend_table.to_excel(writer, sheet_name="Trend", index=False)
    profile_table.to_excel(writer, sheet_name="Profile", index=False)

print("Files exported.")

## Plot 1 — Raw ranking

In [ ]:
plot_rank = ranking_table.sort_values("Raw Crime Value", ascending=True)

plt.figure(figsize=(12, 8))
plt.barh(plot_rank["Country"], plot_rank["Raw Crime Value"], color="steelblue")
plt.title(f"EU Countries Ranked by Raw Crime Value ({latest_year})")
plt.xlabel("Raw Crime Value")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## Plot 2 — Trend factor

In [ ]:
plot_trend = trend_table.sort_values("Trend_%", ascending=True)

plt.figure(figsize=(12, 8))
plt.barh(plot_trend["Country"], plot_trend["Trend_%"], color="steelblue")
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Crime Trend in EU Countries ({start_year}–{latest_year})")
plt.xlabel("Percentage Change (%)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## Plot 3 — Beyond the rankings

In [ ]:
scatter_data = profile_table.copy()

plt.figure(figsize=(12, 8))
plt.scatter(scatter_data["Crime_per_100k"], scatter_data["Trend_%"], color="steelblue")

for _, row in scatter_data.iterrows():
    plt.annotate(
        row["Country"],
        (row["Crime_per_100k"], row["Trend_%"]),
        textcoords="offset points",
        xytext=(3, 3),
        fontsize=7
    )

plt.axhline(0, color="black", linewidth=1)
plt.axvline(scatter_data["Crime_per_100k"].mean(), color="black", linewidth=1, linestyle="--")
plt.title("Crime Profiling of EU Countries: Risk per Person vs Trend")
plt.xlabel("Crime per 100,000 People")
plt.ylabel(f"Trend (% {start_year}–{latest_year})")
plt.tight_layout()
plt.show()